# Hospital Readmission Analysis - Data Understanding

This notebook focuses on understanding the structure, quality, and key characteristics of the dataset before modeling.

We analyze:
- Dataset structure
- Missing values
- Feature relevance
- Initial observations

In [45]:
import pandas as pd

### Loading Dataset

We load the diabetes dataset and inspect its structure.

In [46]:
df = pd.read_csv("../data/raw/diabetic_data.csv")
df.head(3)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


In [47]:
df.shape

(101766, 50)

In [48]:
df.dtypes.value_counts()

str      37
int64    13
Name: count, dtype: int64

### Dataset Overview

The dataset contains hospital encounter records for diabetic patients collected from 130 US hospitals between 1999 and 2008.

Each row represents a hospital encounter, not a unique patient. This means a single patient may appear multiple times across different encounters.

The dataset includes patient demographics, admission details, diagnosis information, and treatment-related variables.

 The variables include both numerical and categorical data types, with several features stored as object type due to categorical values and missing entries.

In [49]:
df['readmitted'].value_counts(dropna=False)

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

## Target Variable

The target variable is `readmitted`, which indicates whether a patient was readmitted to the hospital.

The possible values are:
- `<30`: readmitted within 30 days
- `>30`: readmitted after 30 days
- `NO`: no readmission

For this project, the goal is to predict whether a patient will be readmitted within 30 days.

In [50]:
df.isna().sum().sort_values(ascending=False).head(15)

max_glu_serum               96420
A1Cresult                   84748
race                            0
gender                          0
age                             0
weight                          0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                      0
medical_specialty               0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
dtype: int64

In [51]:
(df == '?').sum().sort_values(ascending=False).head(20)

weight                      98569
medical_specialty           49949
payer_code                  40256
race                         2273
diag_3                       1423
diag_2                        358
diag_1                         21
admission_type_id               0
patient_nbr                     0
encounter_id                    0
time_in_hospital                0
admission_source_id             0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
discharge_disposition_id        0
gender                          0
age                             0
number_inpatient                0
number_emergency                0
dtype: int64

### Handling Inconsistent Missing Values

During initial exploration, we observed that missing values in the dataset are represented in two different ways:

- As `'?'` (string values)
- As standard `NaN` values

These need to be handled together to correctly assess data quality.

We first examined both types separately and found that relying only on `isna()` would underestimate missing values.

To ensure consistency, we converted all `'?'` entries into proper missing values (`NaN`) before performing further analysis.

In [52]:
df = df.replace("?", pd.NA)
df.isna().sum().sort_values(ascending=False).head(10)

weight               98569
max_glu_serum        96420
A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
patient_nbr              0
dtype: int64

In [53]:
(df.isna().sum() / len(df) * 100).sort_values(ascending=False).head(15)

weight                 96.858479
max_glu_serum          94.746772
A1Cresult              83.277322
medical_specialty      49.082208
payer_code             39.557416
race                    2.233555
diag_3                  1.398306
diag_2                  0.351787
diag_1                  0.020636
patient_nbr             0.000000
time_in_hospital        0.000000
admission_source_id     0.000000
num_lab_procedures      0.000000
encounter_id            0.000000
admission_type_id       0.000000
dtype: float64

### Missing Value Analysis

After converting all `'?'` values to proper missing values (`NaN`), we analyzed the percentage of missing data across all features.

Key observations:

- `weight` has extremely high missing values (~97%), making it unreliable for analysis.
- `max_glu_serum` (~95%) and `A1Cresult` (~83%) also have very high missingness and provide limited usable information.
- `payer_code` has a significant portion of missing values (~40%) and is not directly relevant to the prediction task.
- `medical_specialty` has moderate missing values (~49%), but may still contain useful information and will be handled separately.
- `race` has very low missing values (~2%) and can be retained for analysis.
- Diagnosis-related features (`diag_1`, `diag_2`, `diag_3`) have minimal missing values and are important for understanding patient conditions.

Based on this analysis, columns with extremely high missing values will be removed, while others will be handled using appropriate strategies.

### Dropping High-Missing Columns

Based on the missing value analysis, a few columns were identified for removal during preprocessing due to extremely high missingness or limited usefulness for the current project.

Dropped columns:
- `weight`
- `payer_code`
- `max_glu_serum`
- `A1Cresult`

These columns either had very large proportions of missing values or were not essential enough to justify more complex handling at this stage.

### Exploring `medical_specialty`

The `medical_specialty` column was examined to determine whether it could be used meaningfully in the analysis. We checked its missingness, number of unique categories, and category distribution before deciding how it should be handled later during preprocessing.

In [54]:
df['medical_specialty'].isna().sum()

np.int64(49949)

In [55]:
df['medical_specialty'].nunique()

72

In [56]:
df['medical_specialty'].value_counts(normalize=True, dropna=False).head(20) * 100

medical_specialty
NaN                                  49.082208
InternalMedicine                     14.381031
Emergency/Trauma                      7.433720
Family/GeneralPractice                7.310890
Cardiology                            5.259124
Surgery-General                       3.045221
Nephrology                            1.585009
Orthopedics                           1.375705
Orthopedics-Reconstructive            1.211603
Radiologist                           1.120217
Pulmonology                           0.855885
Psychiatry                            0.839180
Urology                               0.673113
ObstetricsandGynecology               0.659356
Surgery-Cardiovascular/Thoracic       0.640685
Gastroenterology                      0.554213
Surgery-Vascular                      0.523751
Surgery-Neuro                         0.459879
PhysicalMedicineandRehabilitation     0.384215
Oncology                              0.341961
Name: proportion, dtype: float64

This notebook focused on understanding the structure and data quality of the hospital readmission dataset before preprocessing. Missing values were examined carefully, including values stored as `?`, and several columns were identified as having very high missingness. The `medical_specialty` feature was also explored and found to be sparse, suggesting that grouped handling may be more appropriate in the preprocessing stage. The next notebook will focus on data cleaning and feature preparation for modeling.